# GFlowNet Molecule Generation & Analysis

This notebook provides an interactive workflow to generate, inspect, and analyze molecules produced by a trained GFlowNet using a proxy property model.

It allows you to:

- Load a trained GFlowNet from an experiment log (configuration and model weights)
- Sample new molecular designs over multiple stochastic rounds
- Evaluate each design using the trained proxy model
- Explore results interactively, including:
  - per-molecule inspection (structure and property value)
  - top-K selection based on the optimization objective
  - summary statistics (mean, standard deviation, min, max)
  - grid visualization of selected designs with hover metadata
- Export generated designs as a CSV file for downstream analysis

All inputs (experiment ID, proxy model paths, sampling settings) are provided through widgets, and results update automatically after generation. 
To get started, **Run All Cells** once. After that, use the widgets below to generate and explore molecules.

*Tip: Hover over molecule tiles to view the SMILES string and property value.*


## Code

In [1]:
import random
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from omegaconf import OmegaConf
from rdkit import Chem
from rdkit.Chem import  DataStructs, rdMolDescriptors, Descriptors, Draw
import torch_geometric.data as gd
from gflownet.proxy.mol_utils import smiles2graph
from gflownet.utils.conditioning import TemperatureConditional
from gflownet.models.graph_transformer import GraphTransformerGFN
from gflownet.algo.trajectory_balance import TrajectoryBalance
from gflownet.algo.flow_matching import FlowMatching
from gflownet.envs.graph_building_env import GraphBuildingEnv
from gflownet.envs.frag_mol_env import FragMolBuildingEnvContext
from gflownet.models import bengio2021flow
from gflownet.proxy.model import load_proxy_to_gflow
import warnings
from scipy.stats import pearsonr, spearmanr
from rdkit import RDLogger   
import ipywidgets as widgets
from IPython.display import display, clear_output
import html
import io
import base64

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings("ignore", category=DeprecationWarning)


# --------------------------
# Generation UI
# --------------------------
STATUS = widgets.Output()
OUT = widgets.Output()

def help_text(text):
    return widgets.HTML(f"<i style='color:#555;font-size:13px'>{text}</i>")

LABEL_STYLE = {"description_width": "140px"}

LOG_ID = widgets.Text(
    value="example",
    description="GFlowNet log ID:",
    style=LABEL_STYLE,
    layout=widgets.Layout(width="420px"),
)

PROXY_PARAMS = widgets.Text(
    value="src/gflownet/proxy/model_params.txt",
    description="Proxy params file:",
    style=LABEL_STYLE,
    layout=widgets.Layout(width="420px"),
)

PROXY_MODEL = widgets.Text(
    value="src/gflownet/proxy/best_model.pt",
    description="Proxy model weights:",
    style=LABEL_STYLE,
    layout=widgets.Layout(width="420px"),
)

NUM_GEN = widgets.BoundedIntText(
    value=50,
    min=1,
    max=10000,
    description="Molecules / round:",
    style=LABEL_STYLE,
    layout=widgets.Layout(width="250px"),
)

REPEATS = widgets.BoundedIntText(
    value=5,
    min=1,
    max=200,
    description="Number of rounds:",
    style=LABEL_STYLE,
    layout=widgets.Layout(width="250px"),
)

SEED = widgets.IntText(
    value=42,
    description="Random seed:",
    style=LABEL_STYLE,
    layout=widgets.Layout(width="250px"),
)

RUN_BTN = widgets.Button(
    description="Run generation",
    button_style="primary",
    layout=widgets.Layout(width="200px"),
)

DOWNLOAD_BTN = widgets.Button(
    description="Download CSV",
    button_style="success",
    disabled=True,
    layout=widgets.Layout(width="200px"),
)

HELP_LOG_ID = help_text(
    "Folder name under src/gflownet/tasks/logs/<ID>/ (must contain config.yaml and model_state.pt)."
)
HELP_PROXY_PARAMS = help_text("Parameter file written by train.py (usually model_params.txt).")
HELP_PROXY_MODEL = help_text("Trained proxy weights written by train.py (usually best_model.pt).")
HELP_NUM_GEN = help_text("Molecules generated per sampling round. Total = num_gen × rounds.")
HELP_REPEATS = help_text("How many independent sampling rounds to run.")
HELP_SEED = help_text("Seed for reproducibility. Same seed → same sampling behavior.")

num_gen = None
repeats = None
seed = None

def _run_generation(_):
    global df_all, num_gen, repeats, seed  # exported

    RUN_BTN.disabled = True
    DOWNLOAD_BTN.disabled = True

    with STATUS:
        clear_output()
        display(widgets.HTML("<b>Status:</b> Running…"))

    with OUT:
        clear_output()

        try:
            run_id = LOG_ID.value.strip()
            if not run_id:
                print("Please provide a Log ID (e.g. 'example').")
                return

            num_gen = int(NUM_GEN.value)
            repeats = int(REPEATS.value)
            seed = int(SEED.value)

            current_dir = Path.cwd()

            proxy_model = load_proxy_to_gflow(PROXY_PARAMS.value, PROXY_MODEL.value)

            yaml_dir = current_dir / "src" / "gflownet" / "tasks" / "logs" / run_id / "config.yaml"
            model_dir = current_dir / "src" / "gflownet" / "tasks" / "logs" / run_id / "model_state.pt"

            if not yaml_dir.exists():
                print(f"config.yaml not found:\n{yaml_dir}")
                return
            if not model_dir.exists():
                print(f"model_state.pt not found:\n{model_dir}")
                return

            cfg = OmegaConf.load(yaml_dir)

            env = GraphBuildingEnv()
            temp_cond = TemperatureConditional(cfg)
            num_cond_dim = temp_cond.encoding_size()

            ctx = FragMolBuildingEnvContext(
                max_frags=cfg.algo.max_nodes,
                num_cond_dim=num_cond_dim,
                fragments=bengio2021flow.FRAGMENTS,
            )

            model = GraphTransformerGFN(
                env_ctx=ctx,
                cfg=cfg,
                num_graph_out=cfg.algo.tb.do_predict_n + 1,
                do_bck=cfg.algo.tb.do_parameterize_p_b,
            )
            model.load_state_dict((torch.load(model_dir)["models_state_dict"][0]))
            model.eval()

            algo = TrajectoryBalance(env, ctx, cfg)

            total_smiles, total_preds = [], []

            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)

            for r in range(repeats):
                with STATUS:
                    clear_output()
                    display(widgets.HTML(f"<b>Status:</b> Running… (round {r+1}/{repeats})"))

                cond_info = temp_cond.sample(num_gen)["encoding"]
                samples = algo.create_training_data_from_own_samples(model=model, n=num_gen, cond_info=cond_info)

                trajectories = [sample["traj"] for sample in samples]
                rdkit_mols = [ctx.graph_to_obj(traj[-1][0]) for traj in trajectories]
                smiles = [Chem.MolToSmiles(mol) for mol in rdkit_mols]

                graphs = [smiles2graph(Chem.MolFromSmiles(smile)) for smile in smiles]
                batch = gd.Batch.from_data_list([g for g in graphs if g is not None])

                preds = (
                    proxy_model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                    .squeeze(dim=-1)
                    .cpu()
                    .detach()
                    .numpy()
                )

                total_smiles.extend(smiles)
                total_preds.extend(preds)

            df_all = pd.DataFrame({"smiles": total_smiles, "reward": total_preds})

            # auto-refresh viewers
            viewer_refresh()
            grid_refresh()

            display(df_all.head(10))
            print(f"Generated: {len(df_all)} molecules (num_gen={num_gen}, rounds={repeats})")
            print("Exported DataFrame: df_all")

            DOWNLOAD_BTN.disabled = False

        finally:
            RUN_BTN.disabled = False
            with STATUS:
                clear_output()
                display(widgets.HTML("<b>Status:</b> Done ✅"))

def _download_csv(_):
    if "df_all" not in globals():
        with STATUS:
            clear_output()
            display(widgets.HTML("<b>Status:</b> No data to download yet."))
        return

    filename = f"gflownet_generated_{LOG_ID.value.strip() or 'results'}.csv"
    df_all.to_csv(filename, index=False)

    with STATUS:
        clear_output()
        display(widgets.HTML(f"<b>Status:</b> CSV saved as <code>{filename}</code>"))

RUN_BTN.on_click(_run_generation)
DOWNLOAD_BTN.on_click(_download_csv)

GEN_UI = widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:5px'>GFlowNet generation inputs</h3>"),
    LOG_ID, HELP_LOG_ID,
    widgets.HTML("<hr style='margin:8px 0'>"),
    PROXY_PARAMS, HELP_PROXY_PARAMS,
    PROXY_MODEL, HELP_PROXY_MODEL,
    widgets.HTML("<hr style='margin:8px 0'>"),
    widgets.HBox([NUM_GEN, REPEATS]),
    widgets.VBox([HELP_NUM_GEN, HELP_REPEATS]),
    widgets.HTML("<hr style='margin:8px 0'>"),
    SEED, HELP_SEED,
    widgets.HTML("<hr style='margin:10px 0'>"),
    widgets.HBox([RUN_BTN, DOWNLOAD_BTN]),
    STATUS,
    OUT,
])


# --------------------------
# Viewer UI (All vs Top-K)
# --------------------------
VIEW_STATE = {"df_view": pd.DataFrame(columns=["smiles", "reward"])}

def df_all_safe():
    df = globals().get("df_all", None)
    if isinstance(df, pd.DataFrame):
        out = df.copy()
        if "smiles" not in out.columns:
            out["smiles"] = None
        if "reward" not in out.columns:
            out["reward"] = None
        return out
    return pd.DataFrame(columns=["smiles", "reward"])

def clean_view_df(df):
    out = df.copy()
    out["reward"] = pd.to_numeric(out.get("reward"), errors="coerce")
    out = out.dropna(subset=["smiles"]).reset_index(drop=True)
    return out

def make_options(df_view, max_items=5000):
    n = min(len(df_view), max_items)
    opts = []
    for i in range(n):
        smi = df_view.loc[i, "smiles"]
        rew = df_view.loc[i, "reward"]
        rew_str = f"{rew:.4f}" if pd.notna(rew) else "nan"
        opts.append((f"[{i}] reward={rew_str} | {smi}", i))
    return opts

def topk_df(df, k, direction):
    out = df.copy()
    out["reward"] = pd.to_numeric(out.get("reward"), errors="coerce")
    out = out.dropna(subset=["smiles", "reward"]).reset_index(drop=True)
    if len(out) == 0:
        return out
    asc = True if direction == "min" else False
    return out.sort_values("reward", ascending=asc).head(int(k)).reset_index(drop=True)

def stats_html(df, title):
    if df is None or len(df) == 0:
        return widgets.HTML(f"<b>{title}</b><br><i style='color:#555'>No molecules available.</i>")
    vals = pd.to_numeric(df.get("reward"), errors="coerce").dropna()
    if len(vals) == 0:
        return widgets.HTML(f"<b>{title}</b><br><i style='color:#555'>No numeric rewards available.</i>")
    mean = float(vals.mean())
    std  = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
    vmin = float(vals.min())
    vmax = float(vals.max())
    n = int(len(vals))
    return widgets.HTML(
        f"<b>{title}</b>"
        f"<div style='margin-top:4px; font-size:13px; color:#333'>"
        f"<span style='display:inline-block; width:120px'><i>Count</i>:</span> {n}<br>"
        f"<span style='display:inline-block; width:120px'><i>Mean</i>:</span> {mean:.6f}<br>"
        f"<span style='display:inline-block; width:120px'><i>Std</i>:</span> {std:.6f}<br>"
        f"<span style='display:inline-block; width:120px'><i>Min</i>:</span> {vmin:.6f}<br>"
        f"<span style='display:inline-block; width:120px'><i>Max</i>:</span> {vmax:.6f}<br>"
        f"</div>"
    )

VIEWER_OUT = widgets.Output()
VIEWER_STATS = widgets.Output()

VIEW_SOURCE = widgets.Dropdown(
    options=[("All molecules (df_all)", "all"), ("Top-K (best rewards)", "topk")],
    value="all",
    description="View:",
    style=LABEL_STYLE,
    layout=widgets.Layout(width="350px"),
)

VIEW_TOPK = widgets.BoundedIntText(
    value=100, min=1, max=50000,
    description="Top-K:",
    style=LABEL_STYLE,
    layout=widgets.Layout(width="250px"),
)

VIEW_DIR = widgets.Dropdown(
    options=[("Minimize (smaller is better)", "min"), ("Maximize (larger is better)", "max")],
    value="min",
    description="Objective:",
    style=LABEL_STYLE,
    layout=widgets.Layout(width="350px"),
)

VIEW_DD = widgets.Dropdown(
    options=[("No molecules yet (run generation first)", None)],
    value=None,
    description="Select molecule:",
    style=LABEL_STYLE,
    layout=widgets.Layout(width="900px"),
)

VIEW_SIZE = widgets.IntSlider(
    value=350, min=200, max=800, step=50,
    description="2D size:",
    style=LABEL_STYLE,
    layout=widgets.Layout(width="500px"),
)

VIEW_REFRESH = widgets.Button(
    description="Refresh list",
    button_style="info",
    layout=widgets.Layout(width="200px"),
)

def viewer_show(_=None):
    with VIEWER_OUT:
        clear_output()
        dfv = VIEW_STATE["df_view"]
        if len(dfv) == 0 or VIEW_DD.value is None:
            display(widgets.HTML("<i style='color:#555'>No molecules loaded yet. Run generation, then click “Refresh list”.</i>"))
            return

        i = VIEW_DD.value
        smi = dfv.loc[i, "smiles"]
        rew = dfv.loc[i, "reward"]
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            print("Invalid SMILES:", smi)
            return

        rew_str = f"{rew:.6f}" if pd.notna(rew) else "nan"
        display(widgets.HTML(f"<b>Reward:</b> {rew_str}<br><b>SMILES:</b> <code>{smi}</code>"))
        display(Draw.MolToImage(mol, size=(VIEW_SIZE.value, VIEW_SIZE.value)))

def viewer_refresh(_=None):
    df = clean_view_df(df_all_safe())

    if VIEW_SOURCE.value == "topk":
        dfv = topk_df(df, k=int(VIEW_TOPK.value), direction=VIEW_DIR.value)
        title = f"Top-{int(VIEW_TOPK.value)} summary ({VIEW_DIR.value})"
    else:
        dfv = df
        title = "All molecules summary"

    VIEW_STATE["df_view"] = dfv

    opts = make_options(dfv, max_items=5000)
    if len(opts) == 0:
        VIEW_DD.options = [("No molecules yet (run generation first)", None)]
        VIEW_DD.value = None
    else:
        VIEW_DD.options = opts
        VIEW_DD.value = opts[0][1]

    with VIEWER_STATS:
        clear_output()
        display(stats_html(dfv.dropna(subset=["reward"]), title))

    viewer_show()

VIEW_SOURCE.observe(viewer_refresh, names="value")
VIEW_TOPK.observe(viewer_refresh, names="value")
VIEW_DIR.observe(viewer_refresh, names="value")
VIEW_DD.observe(viewer_show, names="value")
VIEW_SIZE.observe(viewer_show, names="value")
VIEW_REFRESH.on_click(viewer_refresh)

VIEWER_UI = widgets.VBox([
    widgets.HTML("<h4 style='margin:0 0 6px 0'>Molecule viewer</h4>"),
    widgets.HBox([VIEW_SOURCE, VIEW_TOPK, VIEW_DIR, VIEW_REFRESH]),
    VIEWER_STATS,
    VIEW_DD,
    VIEW_SIZE,
    VIEWER_OUT
])


# --------------------------
# Top-K hover grid UI
# --------------------------
GRID_POOL = {"df": pd.DataFrame(columns=["smiles", "reward"])}
GRID_OUT = widgets.Output()
GRID_STATS = widgets.Output()

def png_uri(pil_img) -> str:
    buf = io.BytesIO()
    pil_img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    return f"data:image/png;base64,{b64}"

GRID_K = widgets.BoundedIntText(
    value=100, min=1, max=5000,
    description="Top K:",
    style=LABEL_STYLE,
    layout=widgets.Layout(width="260px"),
)

GRID_DIR = widgets.Dropdown(
    options=[("Minimize (lower is better)", "min"), ("Maximize (higher is better)", "max")],
    value="min",
    description="Objective:",
    style=LABEL_STYLE,
    layout=widgets.Layout(width="420px"),
)

GRID_REFRESH = widgets.Button(
    description="Refresh Top-K list",
    button_style="info",
    layout=widgets.Layout(width="220px"),
)

GRID_SELECT = widgets.SelectMultiple(
    options=[],
    value=(),
    description="Pick designs:",
    style=LABEL_STYLE,
    layout=widgets.Layout(width="950px", height="220px"),
)

GRID_COUNTER = widgets.HTML("<i style='color:#555'>Selected: 0 molecule(s)</i>")

GRID_HINT = widgets.HTML(
    "<i style='color:#666;font-size:13px'>Tip: Hover over a molecule to see its SMILES and property value.</i>"
)

GRID_TILE = widgets.IntSlider(
    value=220, min=150, max=420, step=10,
    description="Tile size:",
    style=LABEL_STYLE,
    layout=widgets.Layout(width="450px"),
)

GRID_COLS = widgets.BoundedIntText(
    value=4, min=1, max=8,
    description="Columns:",
    style=LABEL_STYLE,
    layout=widgets.Layout(width="260px"),
)

GRID_SHOW = widgets.Button(
    description="Show grid",
    button_style="primary",
    layout=widgets.Layout(width="240px"),
)

def grid_count(_=None):
    n = len(GRID_SELECT.value) if GRID_SELECT.value is not None else 0
    GRID_COUNTER.value = f"<i style='color:#555'>Selected: <b>{n}</b> molecule(s)</i>"

GRID_SELECT.observe(grid_count, names="value")
grid_count()

def grid_refresh(_=None):
    df = clean_view_df(df_all_safe())
    if len(df) == 0:
        GRID_POOL["df"] = df
        GRID_SELECT.options = [("df_all is empty (run generation first)", None)]
        GRID_SELECT.value = ()
        grid_count()
        with GRID_STATS:
            clear_output()
            display(stats_html(pd.DataFrame(columns=["reward"]), "Top-K summary"))
        with GRID_OUT:
            clear_output()
            display(widgets.HTML("<i style='color:#555'>df_all is empty. Run generation first.</i>"))
        return

    dfk = topk_df(df, k=int(GRID_K.value), direction=GRID_DIR.value)
    GRID_POOL["df"] = dfk

    opts = []
    for i in range(len(dfk)):
        smi = str(dfk.loc[i, "smiles"])
        rew = dfk.loc[i, "reward"]
        rew_str = f"{float(rew):.6f}" if pd.notna(rew) else "nan"
        opts.append((f"[{i}] reward={rew_str} | {smi}", i))

    if len(opts) == 0:
        GRID_SELECT.options = [("No valid reward values in df_all", None)]
        GRID_SELECT.value = ()
    else:
        GRID_SELECT.options = opts
        GRID_SELECT.value = tuple(range(min(8, len(opts))))

    grid_count()

    with GRID_STATS:
        clear_output()
        display(stats_html(dfk, f"Top-{len(dfk)} summary ({GRID_DIR.value})"))

    with GRID_OUT:
        clear_output()
        display(widgets.HTML(
            f"<i style='color:#555'>Loaded Top-{len(dfk)} designs. Select designs and click <b>Show grid</b>.</i>"
        ))

def grid_plot(_=None):
    with GRID_OUT:
        clear_output()

        topk = GRID_POOL["df"]
        if len(topk) == 0:
            display(widgets.HTML("<i style='color:#555'>Top-K pool is empty. Click “Refresh Top-K list” first.</i>"))
            return

        idxs = [i for i in list(GRID_SELECT.value) if i is not None]
        if len(idxs) == 0:
            display(widgets.HTML("<i style='color:#555'>Select at least 1 design.</i>"))
            return

        tile_px = int(GRID_TILE.value)
        cols = int(GRID_COLS.value)

        tiles_html = []
        valid = 0

        for i in idxs:
            if i < 0 or i >= len(topk):
                continue

            smi = str(topk.loc[i, "smiles"])
            rew = topk.loc[i, "reward"]

            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                continue

            pil = Draw.MolToImage(mol, size=(tile_px, tile_px))
            src = png_uri(pil)

            tooltip = f"reward = {float(rew):.6f}\nSMILES = {smi}"
            tooltip_escaped = html.escape(tooltip)

            tiles_html.append(f"""
            <div title="{tooltip_escaped}"
                 style="
                    width:{tile_px}px;
                    height:{tile_px}px;
                    border:1px solid #ddd;
                    border-radius:10px;
                    padding:6px;
                    background:white;
                    display:flex;
                    align-items:center;
                    justify-content:center;
                    box-shadow:0 1px 2px rgba(0,0,0,0.06);
                    cursor:help;">
              <img src="{src}" style="width:{tile_px}px;height:{tile_px}px;" />
            </div>
            """)
            valid += 1

        if valid == 0:
            display(widgets.HTML("<i style='color:#555'>No valid molecules among your selection.</i>"))
            return

        grid_html = f"""
        {GRID_HINT.value}
        <div style="
            display:grid;
            grid-template-columns: repeat({cols}, {tile_px+14}px);
            gap:14px;
            align-items:start;">
            {''.join(tiles_html)}
        </div>
        """
        display(widgets.HTML(grid_html))

GRID_REFRESH.on_click(grid_refresh)
GRID_SHOW.on_click(grid_plot)

GRID_UI = widgets.VBox([
    widgets.HTML("<h4 style='margin:0 0 8px 0'>Top-K grid viewer</h4>"),
    widgets.HBox([GRID_K, GRID_DIR, GRID_REFRESH]),
    GRID_STATS,
    GRID_SELECT,
    GRID_COUNTER,
    widgets.HBox([GRID_TILE, GRID_COLS, GRID_SHOW]),
    GRID_OUT
])


## Run

In [2]:
display(GEN_UI)

In [3]:
display(VIEWER_UI)

In [4]:
display(GRID_UI)